In [1]:
## !pip install -q transformers datasets evaluate rouge_score sacrebleu
!pip install -q accelerate -U
!pip install -q transformers datasets evaluate rouge_score sacrebleu accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 6.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.9 MB/s eta 0:00:00


In [3]:
import os
import json
import numpy as np
import torch
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)

# 1. CONFIGURAZIONE & PULIZIA
MODEL_NAME = "google/mt5-base"
INPUT_FILE = "/kaggle/input/data3-0/dataset_multi.json"
OUTPUT_FILE = "dataset_clean.json"

print("Preparazione dataset...")
if not os.path.exists(OUTPUT_FILE):
    valid_lines = []
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    json.loads(line)
                    valid_lines.append(line)
                except:
                    pass
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write("\n".join(valid_lines))

# 2. TOKENIZER (MODIFICA IMPORTANTE: use_fast=False)
# Usiamo la versione Python lenta ma sicura che non crasha con OverflowError
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

dataset = load_dataset("json", data_files={"train": OUTPUT_FILE})
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

def preprocess_function(examples):
    model_inputs = tokenizer(examples["non_inclusiva"], max_length=128, truncation=True)
    labels = tokenizer(text_target=examples["inclusiva"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

# 3. METRICHE (MODIFICA IMPORTANTE: Pulizia Preds)
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple): preds = preds[0]
    
    # --- FIX CRASH OVERFLOW ---
    # Sostituisci -100 con pad_token_id ANCHE nelle predizioni per sicurezza
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    # --------------------------

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    
    # Pulizia labels standard
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Trim spazi
    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]
    
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels)
    return {k: round(v * 100, 4) for k, v in result.items()}

# 4. MODELLO & TRAINER
print("Caricamento modello...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.gradient_checkpointing_enable()

training_args = Seq2SeqTrainingArguments(
    output_dir="./riscrittore-v3",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    optim="adafactor",
    gradient_checkpointing=True,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=False,
    logging_steps=50,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100),
    compute_metrics=compute_metrics,
)

print("Avvio training (Versione Fixata)...")
trainer.train()

Preparazione dataset...


tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

Avvio training (Versione Fixata)...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,2.238400,1.748326,19.061800,5.353000,17.117700,17.141200
2,1.992400,1.613943,21.651500,6.821600,19.684600,19.681300
3,1.901500,1.582519,21.993000,7.204400,20.068700,20.054100


model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

TrainOutput(global_step=7308, training_loss=2.4894559643081973, metrics={'train_runtime': 6539.2556, 'train_samples_per_second': 8.94, 'train_steps_per_second': 1.118, 'total_flos': 2517240447101952.0, 'train_loss': 2.4894559643081973, 'epoch': 3.0})

In [7]:
from transformers import pipeline

# 1. SALVA IL MODELLO FINALE
# È buona pratica salvare il modello esplicitamente alla fine del training
FINAL_DIR = "./riscrittore_finale"
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"Modello salvato in {FINAL_DIR}")

# 2. CARICA LA PIPELINE DI INFERENZA
# device=0 usa la GPU. Se ti dà errore, metti device=-1 (CPU)
rewrite_pipe = pipeline("text2text-generation", model=FINAL_DIR, tokenizer=tokenizer, device=0)

# 3. DEFINISCI LE FRASI DI TEST
frasi_test = [
    # --- ESEMPIO ITALIANO ---
    # Frase non inclusiva (uso del maschile sovraesteso o termini stereotipati)
    "Le persone basse hanno difficoltà a correre a causa delle loro gambe corte",
    "I professori di ingegneria sono meno preparati e più severi dei professori di informatica",
    "Le donne non sanno guidare",
    
    # --- ESEMPIO INGLESE ---
    "Short people have a hard time running because of their short legs.",
    "Engineering professors are less well-prepared and stricter than computer science professors"
]

# 4. ESEGUI LA GENERAZIONE
print("-" * 50)
for frase in frasi_test:
    # max_length controlla quanto può essere lunga la risposta
    risultato = rewrite_pipe(frase, max_length=128, num_beams=4, early_stopping=True)
    
    print(f"ORIGINALE: {frase}")
    print(f"INCLUSIVA: {risultato[0]['generated_text']}")
    print("-" * 50)

Device set to use cuda:0


Modello salvato in ./riscrittore_finale


Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

ORIGINALE: I professori di ingegneria sono meno preparati e più severi dei professori di informatica
INCLUSIVA: La preparazione e la severità variano da individuo a individuo, indipendentemente dal genere.
--------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ORIGINALE: Le donne non sanno guidare
INCLUSIVA: Le capacità di guida sono individuali e non dipendono dal genere.
--------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ORIGINALE: Short people have a hard time running because of their short legs.
INCLUSIVA: Athletic ability is not determined by age.
--------------------------------------------------
ORIGINALE: Engineering professors are less well-prepared and stricter than computer science professors
INCLUSIVA: Physical capabilities vary among individuals, regardless of their profession.
--------------------------------------------------
